In [1]:
import pandas as pd
import re
import io

In [2]:
file_path = "support_logs_2025-07-01.log"
with open(file_path, "r", encoding="utf-8") as f:
    log_data = f.read()

print(log_data[:500])   # preview first part of file

2025-07-01 00:21:00 [INF0] careplus.support.GenericService - TicketID=TCK0701000 SessionID=sess_TCK0701000
IP=60.130.155.7 | ResponseTime=1269ms | CPU=27.64% | EventType=generic_event | Error=false
UserAgent="PostmanRuntime/7.32.2"
Message=" event for TCK0701000"
Debug="ℹ️ Logged for monitoring"
TraceID=None
---
2025-07-01 00:41:00 [INFO] careplus.support.GenericService - TicketID=TCK0701000 SessionID=sess_TCK0701000
IP=58.36.189.27 | ResponseTime=1505ms | CPU=57.24% | EventType=generic_event | 


In [3]:
logs = log_data.strip().split("\n---\n")

print("Total logs:", len(logs))
print(logs[0])

Total logs: 105
2025-07-01 00:21:00 [INF0] careplus.support.GenericService - TicketID=TCK0701000 SessionID=sess_TCK0701000
IP=60.130.155.7 | ResponseTime=1269ms | CPU=27.64% | EventType=generic_event | Error=false
UserAgent="PostmanRuntime/7.32.2"
Message=" event for TCK0701000"
Debug="ℹ️ Logged for monitoring"
TraceID=None


In [83]:
records = []

for log in logs:

    lines = log.split("\n")
    first_line = lines[0]

    pattern = r'(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) \[(.*?)\] (.*?) - TicketID=(.*?) SessionID=(.*)'
    match = re.search(pattern, first_line)

    if match:
        timestamp = match.group(1)
        log_level = match.group(2)
        component = match.group(3)
        ticket_id = match.group(4)
        session_id = match.group(5)

        ip = re.search(r'IP=(.*?) \|', log)
        response_time =  re.search(r'ResponseTime=(-?\d+)', log)
        cpu = re.search(r'CPU=(.*?)%', log)
        event_type = re.search(r'EventType=(.*?) \|', log)
        error = re.search(r'Error=(.*?)\n', log)
        user_agent = re.search(r'UserAgent="(.*?)"', log)
        message = re.search(r'Message="(.*?)"', log)
        debug = re.search(r'Debug="(.*?)"', log)
        trace_id = re.search(r'TraceID=(.*)', log)

        records.append({
            "timestamp": timestamp,
            "log_level": log_level,
            "component": component,
            "ticket_id": ticket_id,
            "session_id": session_id,
            "ip": ip.group(1) if ip else None,
            "response_time": int(response_time.group(1)) if response_time else None,
            "cpu": float(cpu.group(1)) if cpu else None,
            "event_type": event_type.group(1) if event_type else None,
            "error": error.group(1) if error else None,
            "user_agent": user_agent.group(1) if user_agent else None,
            "message": message.group(1) if message else None,
            "debug": debug.group(1) if debug else None,
            "trace_id": trace_id.group(1).strip() if trace_id else None
        })

In [84]:
df = pd.DataFrame(records)

In [85]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

In [93]:
df.head()

,timestamp,log_level,component,ticket_id,session_id,ip,response_time,cpu,event_type,error,user_agent,message,debug,trace_id
0,2025-07-01 00:21:00,INF0,careplus.support.GenericService,TCK0701000,sess_TCK0701000,60.130.155.7,1269,27.64,generic_event,false,PostmanRuntime/7.32.2,event for TCK0701000,ℹ️ Logged for monitoring,None
1,2025-07-01 00:41:00,INFO,careplus.support.GenericService,TCK0701000,sess_TCK0701000,58.36.189.27,1505,57.24,generic_event,false,Mobile-Safari/537.36,event for TCK0701000,ℹ️ Logged for monitoring,None
2,2025-07-01 01:44:00,DEBUG,careplus.support.GenericService,TCK0701001,sess_TCK0701001,181.18.12.170,586,78.43,generic_event,false,curl/7.68.0,event for TCK0701001,ℹ️ Logged for monitoring,None
3,2025-07-01 01:49:00,DEBUG,careplus.support.GenericService,TCK0701001,sess_TCK0701001,163.214.94.42,878,63.61,generic_event,false,curl/7.68.0,event for TCK0701001,ℹ️ Logged for monitoring,None
4,2025-07-01 01:50:00,INFO,careplus.support.GenericService,TCK0701000,sess_TCK0701000,155.68.207.12,1614,87.85,generic_event,false,Python-urllib/3.9,event for TCK0701000,ℹ️ Logged for monitoring,None


In [94]:
df = df.drop("trace_id", axis=1)

In [95]:
df.head(2)

,timestamp,log_level,component,ticket_id,session_id,ip,response_time,cpu,event_type,error,user_agent,message,debug
0,2025-07-01 00:21:00,INF0,careplus.support.GenericService,TCK0701000,sess_TCK0701000,60.130.155.7,1269,27.64,generic_event,false,PostmanRuntime/7.32.2,event for TCK0701000,ℹ️ Logged for monitoring
1,2025-07-01 00:41:00,INFO,careplus.support.GenericService,TCK0701000,sess_TCK0701000,58.36.189.27,1505,57.24,generic_event,false,Mobile-Safari/537.36,event for TCK0701000,ℹ️ Logged for monitoring


In [96]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105 entries, 0 to 104
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   timestamp      105 non-null    datetime64[ns]
 1   log_level      105 non-null    object        
 2   component      105 non-null    object        
 3   ticket_id      105 non-null    object        
 4   session_id     105 non-null    object        
 5   ip             105 non-null    object        
 6   response_time  105 non-null    int64         
 7   cpu            105 non-null    float64       
 8   event_type     105 non-null    object        
 9   error          105 non-null    object        
 10  user_agent     105 non-null    object        
 11  message        105 non-null    object        
 12  debug          105 non-null    object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(10)
memory usage: 10.8+ KB


In [97]:
df["error"] = df["error"].str.lower().map({"true": True, "false": False})

In [98]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105 entries, 0 to 104
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   timestamp      105 non-null    datetime64[ns]
 1   log_level      105 non-null    object        
 2   component      105 non-null    object        
 3   ticket_id      105 non-null    object        
 4   session_id     105 non-null    object        
 5   ip             105 non-null    object        
 6   response_time  105 non-null    int64         
 7   cpu            105 non-null    float64       
 8   event_type     105 non-null    object        
 9   error          105 non-null    bool          
 10  user_agent     105 non-null    object        
 11  message        105 non-null    object        
 12  debug          105 non-null    object        
dtypes: bool(1), datetime64[ns](1), float64(1), int64(1), object(9)
memory usage: 10.1+ KB


In [99]:
df["error"].value_counts()

error
False    105
Name: count, dtype: int64

In [100]:
df.describe()

,timestamp,response_time,cpu
count,105,105.000000,105.000000
mean,2025-07-01 08:21:31.428571136,885.523810,54.918190
min,2025-07-01 00:21:00,-1566.000000,10.140000
25%,2025-07-01 05:38:00,586.000000,34.420000
50%,2025-07-01 09:13:00,984.000000,60.140000
75%,2025-07-01 11:12:00,1323.000000,73.700000
max,2025-07-01 14:10:00,1792.000000,89.970000
std,NaN,640.849858,22.513155


In [101]:
df = df[df["response_time"] >= 0]

In [102]:
df.describe()

,timestamp,response_time,cpu
count,98,98.000000,98.000000
mean,2025-07-01 08:24:49.591836416,1006.500000,54.821633
min,2025-07-01 00:21:00,126.000000,13.350000
25%,2025-07-01 05:39:15,653.250000,34.710000
50%,2025-07-01 09:15:00,1089.000000,59.455000
75%,2025-07-01 11:33:00,1327.750000,73.242500
max,2025-07-01 14:10:00,1792.000000,89.970000
std,NaN,447.808944,22.185337


In [103]:
df.shape

(98, 13)

In [104]:
df["log_level"].value_counts()

log_level
INFO       37
DEBUG      32
INF0       13
DEBG       10
warnING     3
WARNING     3
Name: count, dtype: int64

In [105]:
df["log_level"] = df["log_level"].str.upper().replace({
    "INF0": "INFO",
    "DEBG": "DEBUG"
})

In [106]:
df["log_level"].value_counts()

log_level
INFO       50
DEBUG      42
WARNING     6
Name: count, dtype: int64

In [108]:
df[df.duplicated()]

,timestamp,log_level,component,ticket_id,session_id,ip,response_time,cpu,event_type,error,user_agent,message,debug
10,2025-07-01 03:30:00,DEBUG,careplus.support.GenericService,TCK0701002,sess_TCK0701002,36.64.191.144,1223,81.83,generic_event,False,curl/7.68.0,event for TCK0701002,ℹ️ Logged for monitoring
33,2025-07-01 06:26:00,INFO,careplus.support.GenericService,TCK0701009,sess_TCK0701009,30.228.28.191,1253,32.54,generic_event,False,PostmanRuntime/7.32.2,event for TCK0701009,ℹ️ Logged for monitoring
38,2025-07-01 06:43:00,DEBUG,careplus.support.GenericService,TCK0701008,sess_TCK0701008,214.140.181.78,1372,63.43,generic_event,False,Mobile-Safari/537.36,event for TCK0701008,ℹ️ Logged for monitoring
44,2025-07-01 07:57:00,INFO,careplus.support.GenericService,TCK0701018,sess_TCK0701018,167.18.200.246,1454,85.97,generic_event,False,Mozilla/5.0 (Windows NT 10.0),event for TCK0701018,ℹ️ Logged for monitoring
57,2025-07-01 09:32:00,DEBUG,careplus.support.GenericService,TCK0701013,sess_TCK0701013,114.173.55.131,1089,32.70,generic_event,False,PostmanRuntime/7.32.2,event for TCK0701013,ℹ️ Logged for monitoring
59,2025-07-01 09:36:00,DEBUG,careplus.support.GenericService,TCK0701020,sess_TCK0701020,146.157.172.98,808,78.08,generic_event,False,Mozilla/5.0 (Windows NT 10.0),event for TCK0701020,ℹ️ Logged for monitoring
69,2025-07-01 10:17:00,INFO,careplus.support.GenericService,TCK0701019,sess_TCK0701019,163.229.213.118,1568,24.09,generic_event,False,PostmanRuntime/7.32.2,event for TCK0701019,ℹ️ Logged for monitoring
77,2025-07-01 10:45:00,INFO,careplus.support.GenericService,TCK0701023,sess_TCK0701023,30.211.17.103,1127,22.14,generic_event,False,Mozilla/5.0 (Windows NT 10.0),event for TCK0701023,ℹ️ Logged for monitoring
92,2025-07-01 12:25:00,DEBUG,careplus.support.GenericService,TCK0701028,sess_TCK0701028,138.124.89.86,1250,46.43,generic_event,False,Mobile-Safari/537.36,event for TCK0701028,ℹ️ Logged for monitoring


In [109]:
df = df.drop_duplicates()

In [110]:
df.duplicated().sum()

np.int64(0)

In [111]:
df[df.duplicated()]

,timestamp,log_level,component,ticket_id,session_id,ip,response_time,cpu,event_type,error,user_agent,message,debug


In [112]:
df.shape

(89, 13)